In [27]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dhruvildave/english-handwritten-characters-dataset")

print("Path to dataset files:", path)

100%|██████████| 13.1M/13.1M [00:00<00:00, 84.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/dhruvildave/english-handwritten-characters-dataset/versions/3


In [28]:
import os

DATASET_PATH = "/root/.cache/kagglehub/datasets/dhruvildave/english-handwritten-characters-dataset/versions/3"

print("Dataset exists:", os.path.exists(DATASET_PATH))
print("\nContents:")
print(os.listdir(DATASET_PATH))

Dataset exists: True

Contents:
['Img', 'english.csv']


In [29]:
import os

for root, dirs, files in os.walk(DATASET_PATH):
    if "english.csv" in files or "Img" in dirs:
        print("ROOT:", root)
        print("Dirs:", dirs[:10])
        print("Files:", files[:10])

ROOT: /root/.cache/kagglehub/datasets/dhruvildave/english-handwritten-characters-dataset/versions/3
Dirs: ['Img']
Files: ['english.csv']


CSV: True
Img folder: False
First image: False


FOUND english.csv at:
/content/english.csv


In [5]:
import argparse
import copy
import json
import os
import random
import time
from collections import defaultdict

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (accuracy_score, auc, classification_report,
                             confusion_matrix, precision_recall_fscore_support,
                             roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize


In [7]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def hid_label(h):
    return "-".join(str(x) for x in h)


def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"  saved {path}")

In [8]:
def load_dataset(data_dir, img_size):
    csv_path = os.path.join(data_dir, "english.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Could not find {csv_path}. Pass --data_dir correctly.")
    df = pd.read_csv(csv_path)
    X = np.zeros((len(df), img_size * img_size), dtype=np.float32)
    for i, rel in enumerate(df["image"]):
        img = Image.open(os.path.join(data_dir, rel)).convert("L")     # grayscale
        img = img.resize((img_size, img_size), Image.BILINEAR)          # resize
        arr = np.asarray(img, dtype=np.float32) / 255.0                 # scale to [0,1]
        X[i] = (1.0 - arr).ravel()                                      # invert (ink=1) + flatten
    le = LabelEncoder()
    y = le.fit_transform(df["label"].astype(str).values)
    return X, y, le.classes_

In [9]:
def save_sample_grid(X, y, classes, img_size, path, n=20):
    idx = np.random.choice(len(X), n, replace=False)
    fig, axes = plt.subplots(2, n // 2, figsize=(1.4 * n // 2, 3.2))
    for ax, i in zip(axes.ravel(), idx):
        ax.imshow(X[i].reshape(img_size, img_size), cmap="gray_r")
        ax.set_title(str(classes[y[i]]), fontsize=9)
        ax.axis("off")
    plt.suptitle("Preprocessed samples (resized, inverted, normalised)")
    savefig(path)


In [10]:
class PerceptronOvR:
    def __init__(self, n_classes, lr=0.1, epochs=100, seed=42):
        self.C, self.lr, self.epochs, self.seed = n_classes, lr, epochs, seed
        self.W = None
        self.history = defaultdict(list)

    @staticmethod
    def _bias(X):
        return np.hstack([X, np.ones((len(X), 1), dtype=X.dtype)])

    def decision_function(self, X):
        return self._bias(X) @ self.W

    def predict(self, X):
        return self.decision_function(X).argmax(1)

    def fit(self, X, y, Xv, yv):
        rng = np.random.default_rng(self.seed)
        Xb, Xvb = self._bias(X), self._bias(Xv)
        n, d = Xb.shape
        T = np.zeros((n, self.C), dtype=np.float32)
        T[np.arange(n), y] = 1.0
        W = np.zeros((d, self.C), dtype=np.float32)
        best_va, best_W = -1.0, W.copy()
        for ep in range(self.epochs):
            n_updates = 0
            for i in rng.permutation(n):
                xi = Xb[i]
                y_hat = (xi @ W >= 0).astype(np.float32)          # step activation
                err = T[i] - y_hat                                # (y - y_hat)
                idx = np.flatnonzero(err)
                if idx.size:
                    W[:, idx] += self.lr * xi[:, None] * err[idx][None, :]
                    n_updates += 1
            tr_scores, va_scores = Xb @ W, Xvb @ W
            tr_acc = accuracy_score(y, tr_scores.argmax(1))
            va_acc = accuracy_score(yv, va_scores.argmax(1))
            self.history["train_err"].append(1 - tr_acc)
            self.history["val_err"].append(1 - va_acc)
            self.history["bit_err"].append(float(((tr_scores >= 0) != (T > 0)).mean()))
            self.history["n_updates"].append(n_updates)
            if va_acc > best_va:
                best_va, best_W = va_acc, W.copy()
            if (ep + 1) % 10 == 0 or ep == 0:
                print(f"    PLA epoch {ep + 1:3d}/{self.epochs}  train_err={1 - tr_acc:.4f}  "
                      f"val_err={1 - va_acc:.4f}  samples_updated={n_updates}")
        self.W = best_W
        return self


In [11]:
ACTS = {"relu": nn.ReLU, "sigmoid": nn.Sigmoid, "tanh": nn.Tanh}


class MLP(nn.Module):
    def __init__(self, in_dim, hidden, n_classes, act="relu", dropout=0.0):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), ACTS[act]()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = h
        layers.append(nn.Linear(d, n_classes))               # logits (softmax is inside the loss)
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                if act == "relu":
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                else:
                    nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)


def compute_loss(logits, y, n_classes, kind):
    if kind == "cross_entropy":
        return F.cross_entropy(logits, y)
    if kind == "mse":                                        # squared error on softmax outputs
        p = F.softmax(logits, dim=1)
        return ((p - F.one_hot(y, n_classes).float()) ** 2).sum(1).mean()
    raise ValueError(kind)


def make_optimizer(cfg, params):
    lr, wd, name = cfg["lr"], cfg["weight_decay"], cfg["optimizer"]
    if name in ("gd", "sgd"):                                # gd = full-batch, sgd = mini-batch
        return torch.optim.SGD(params, lr=lr, weight_decay=wd)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd)
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=wd)
    raise ValueError(name)


@torch.no_grad()
def eval_split(model, X, y, n_classes, loss_kind):
    model.eval()
    logits = model(X)
    loss = compute_loss(logits, y, n_classes, loss_kind).item()
    err = (logits.argmax(1) != y).float().mean().item()
    return loss, err


def train_mlp(cfg, D, epochs, device, seed):
    """Train one MLP. Keeps the weights of the epoch with the best validation accuracy."""
    set_seed(seed)
    C = D["n_classes"]
    model = MLP(D["Xtr"].shape[1], cfg["hidden"], C, cfg["activation"], cfg["dropout"]).to(device)
    opt = make_optimizer(cfg, model.parameters())
    n = len(D["Xtr"])
    bs = n if cfg["optimizer"] == "gd" else cfg["batch_size"]
    hist = defaultdict(list)
    best_acc, best_state, best_ep = -1.0, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            loss = compute_loss(model(D["Xtr"][idx]), D["ytr"][idx], C, cfg["loss"])
            loss.backward()
            opt.step()
        tl, te = eval_split(model, D["Xtr"], D["ytr"], C, cfg["loss"])
        vl, ve = eval_split(model, D["Xva"], D["yva"], C, cfg["loss"])
        hist["train_loss"].append(tl); hist["train_err"].append(te)
        hist["val_loss"].append(vl); hist["val_err"].append(ve)
        if 1 - ve > best_acc:
            best_acc, best_ep = 1 - ve, ep + 1
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, hist, best_acc, best_ep


def cfg_key(cfg):
    return json.dumps(cfg, sort_keys=True)


def run_trial(cfg, D, epochs, device, seed, cache):
    key = cfg_key(cfg)
    if key in cache:
        return cache[key]
    t0 = time.time()
    _, hist, best_acc, best_ep = train_mlp(cfg, D, epochs, device, seed)
    row = {**cfg, "hidden": hid_label(cfg["hidden"]), "n_layers": len(cfg["hidden"]),
           "val_acc": best_acc, "best_epoch": best_ep,
           "train_err_at_best": hist["train_err"][best_ep - 1],
           "seconds": round(time.time() - t0, 1)}
    cache[key] = (row, hist)
    print(f"    [{row['optimizer']:>12s} lr={cfg['lr']:<7g} bs={cfg['batch_size']:<3d} "
          f"{cfg['activation']:<7s} {cfg['loss']:<13s} hid={row['hidden']:<16s} "
          f"do={cfg['dropout']} wd={cfg['weight_decay']:g}] val_acc={best_acc:.4f} "
          f"(ep {best_ep}, {row['seconds']}s)")
    return cache[key]

In [12]:
SPACE = {
    "hidden": [(128,), (256,), (512,), (256, 128), (512, 256), (256, 128, 64)],
    "activation": ["relu", "sigmoid", "tanh"],
    "loss": ["cross_entropy", "mse"],
    "optimizer": ["gd", "sgd", "sgd_momentum", "adam"],
    "lr": [1e-4, 1e-3, 1e-2, 1e-1],
    "batch_size": [16, 32, 64, 128, 256],
}


def random_search(D, n_trials, epochs, device, seed, cache):
    rng = random.Random(seed)
    seen, results = set(), []
    tries = 0
    while len(results) < n_trials and tries < n_trials * 20:
        tries += 1
        cfg = {k: rng.choice(v) for k, v in SPACE.items()}
        cfg.update(dropout=0.0, weight_decay=0.0)
        if cfg_key(cfg) in seen:
            continue
        seen.add(cfg_key(cfg))
        row, _ = run_trial(cfg, D, epochs, device, seed, cache)
        results.append((cfg, row))
    return results


def greedy_stage_tuning(base, D, epochs, device, seed, cache):
    """Stage-wise (coordinate) search: tune one group at a time, keep the best, move on."""
    stages = {}
    records = []

    def record(stage, label, row):
        records.append({"stage": stage, "setting": label, "val_acc": row["val_acc"],
                        "best_epoch": row["best_epoch"], "train_err_at_best": row["train_err_at_best"]})

    # Stage 1: optimizer x learning rate (jointly, because they interact strongly)
    print("\n  Stage 1: optimizer x learning rate")
    grid = {}
    for opt in SPACE["optimizer"]:
        for lr in SPACE["lr"]:
            row, _ = run_trial({**base, "optimizer": opt, "lr": lr}, D, epochs, device, seed, cache)
            grid[(opt, lr)] = row["val_acc"]
            record("optimizer x lr", f"{opt} | lr={lr:g}", row)
    (b_opt, b_lr) = max(grid, key=grid.get)
    base = {**base, "optimizer": b_opt, "lr": b_lr}
    stages["optimizer_lr_grid"] = grid
    print(f"  -> best optimizer={b_opt}, lr={b_lr:g}")

    # Stages 2..8: one parameter at a time
    single = [
        ("batch_size", "batch size", [16, 32, 64, 128, 256]),
        ("activation", "activation", ["relu", "sigmoid", "tanh"]),
        ("loss", "loss function", ["cross_entropy", "mse"]),
        ("hidden", "architecture / depth", [(128,), (256,), (512,), (256, 256), (256, 256, 256),
                                            (256, 256, 256, 256), (512, 256, 128)]),
        ("dropout", "dropout", [0.0, 0.2, 0.3, 0.5]),
        ("weight_decay", "weight decay (L2)", [0.0, 1e-5, 1e-4, 1e-3]),
    ]
    for param, title, values in single:
        print(f"\n  Stage: {title}")
        res = []
        for v in values:
            row, _ = run_trial({**base, param: v}, D, epochs, device, seed, cache)
            label = hid_label(v) if param == "hidden" else str(v)
            res.append((label, row["val_acc"], v))
            record(title, label, row)
        best = max(res, key=lambda t: t[1])
        base = {**base, param: best[2]}
        stages[param] = [(a, b) for a, b, _ in res]
        print(f"  -> best {title} = {best[0]}  (val_acc={best[1]:.4f})")
    return base, stages, pd.DataFrame(records)


def hyperparameter_impact(df):
    rows = []
    for p in ["hidden", "activation", "loss", "optimizer", "lr", "batch_size"]:
        g = df.groupby(p)["val_acc"].mean()
        rows.append({"hyperparameter": p, "mean_val_acc_range": g.max() - g.min(),
                     "best_level": g.idxmax(), "worst_level": g.idxmin()})
    return pd.DataFrame(rows).sort_values("mean_val_acc_range", ascending=False)

In [13]:
def roc_micro_macro(y_true, y_score, n_classes):
    Y = label_binarize(y_true, classes=np.arange(n_classes))
    fpr_mi, tpr_mi, _ = roc_curve(Y.ravel(), y_score.ravel())
    grid, mean_tpr, k = np.linspace(0, 1, 500), np.zeros(500), 0
    for c in range(n_classes):
        if Y[:, c].sum() == 0:
            continue
        f, t, _ = roc_curve(Y[:, c], y_score[:, c])
        mean_tpr += np.interp(grid, f, t)
        k += 1
    mean_tpr /= k
    return (fpr_mi, tpr_mi, auc(fpr_mi, tpr_mi)), (grid, mean_tpr, auc(grid, mean_tpr))


def evaluate_model(name, y_true, y_pred, y_score, classes, out_dir):
    C = len(classes)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

    with open(os.path.join(out_dir, f"classification_report_{name}.txt"), "w") as fh:
        fh.write(classification_report(y_true, y_pred, labels=np.arange(C),
                                       target_names=[str(c) for c in classes], zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(C))
    np.savetxt(os.path.join(out_dir, f"confusion_matrix_{name}.csv"), cm, fmt="%d", delimiter=",")
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    plt.figure(figsize=(11, 10))
    plt.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    plt.colorbar(label="row-normalised fraction")
    plt.xticks(range(C), classes, fontsize=6)
    plt.yticks(range(C), classes, fontsize=6)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"Confusion matrix - {name} (test set, acc={acc:.3f})")
    savefig(os.path.join(out_dir, f"confusion_matrix_{name}.png"))

    micro, macro = roc_micro_macro(y_true, y_score, C)
    plt.figure(figsize=(6, 5))
    plt.plot(micro[0], micro[1], label=f"micro-avg (AUC={micro[2]:.3f})")
    plt.plot(macro[0], macro[1], label=f"macro-avg (AUC={macro[2]:.3f})")
    plt.plot([0, 1], [0, 1], "k--", lw=0.8)
    plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
    plt.title(f"ROC (one-vs-rest) - {name}")
    plt.legend(loc="lower right")
    savefig(os.path.join(out_dir, f"roc_{name}.png"))

    return {"model": name, "accuracy": acc, "precision_macro": p, "recall_macro": r, "f1_macro": f,
            "precision_weighted": pw, "recall_weighted": rw, "f1_weighted": fw,
            "auc_micro": micro[2], "auc_macro": macro[2], "_micro": micro, "_macro": macro}

In [14]:
def plot_pla_convergence(h, path):
    ep = np.arange(1, len(h["train_err"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(ep, h["train_err"], label="train error (argmax)")
    ax[0].plot(ep, h["val_err"], label="validation error")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("error rate"); ax[0].legend()
    ax[0].set_title("PLA: error vs epochs")
    ax[1].plot(ep, h["n_updates"], color="tab:red")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("# samples that triggered an update")
    ax[1].set_title("PLA: weight updates per epoch (does not reach 0 -> not linearly separable)")
    ax[1].title.set_fontsize(9)
    savefig(path)


def plot_optimizer_lr_heatmap(grid, path):
    opts, lrs = SPACE["optimizer"], SPACE["lr"]
    M = np.array([[grid[(o, l)] for l in lrs] for o in opts])
    plt.figure(figsize=(6.5, 4))
    plt.imshow(M, cmap="viridis", aspect="auto")
    plt.colorbar(label="validation accuracy")
    plt.xticks(range(len(lrs)), [f"{l:g}" for l in lrs])
    plt.yticks(range(len(opts)), opts)
    for i in range(len(opts)):
        for j in range(len(lrs)):
            plt.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", color="w", fontsize=9)
    plt.xlabel("learning rate"); plt.ylabel("optimizer")
    plt.title("Tuning: optimizer x learning rate")
    savefig(path)


def plot_stage_bars(stages, path):
    order = [("batch_size", "Batch size"), ("activation", "Activation"), ("loss", "Loss function"),
             ("hidden", "Hidden layers (architecture)"), ("dropout", "Dropout"),
             ("weight_decay", "Weight decay")]
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    for ax, (k, t) in zip(axes.ravel(), order):
        labels, vals = zip(*stages[k])
        ax.bar(range(len(vals)), vals, color="tab:blue")
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
        ax.set_ylim(max(0, min(vals) - 0.05), min(1, max(vals) + 0.03))
        ax.set_title(t); ax.set_ylabel("val accuracy")
    savefig(path)


def plot_impact(impact, path):
    plt.figure(figsize=(6, 4))
    plt.barh(impact["hyperparameter"][::-1], impact["mean_val_acc_range"][::-1], color="tab:orange")
    plt.xlabel("max - min of mean validation accuracy across levels")
    plt.title("Random search: hyper-parameter impact")
    savefig(path)


def plot_optimizer_convergence(cache, base, grid, path):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    for opt in SPACE["optimizer"]:
        best_lr = max(SPACE["lr"], key=lambda l: grid[(opt, l)])
        _, h = cache[cfg_key({**base, "optimizer": opt, "lr": best_lr})]
        ep = np.arange(1, len(h["train_err"]) + 1)
        ax[0].plot(ep, h["train_loss"], label=f"{opt} (lr={best_lr:g})")
        ax[1].plot(ep, h["train_err"], label=f"{opt} (lr={best_lr:g})")
    ax[0].set_yscale("log"); ax[0].set_title("Training loss vs epochs (best lr per optimizer)")
    ax[1].set_title("Training error vs epochs (best lr per optimizer)")
    for a in ax:
        a.set_xlabel("epoch"); a.legend()
    ax[0].set_ylabel("training loss"); ax[1].set_ylabel("training error")
    savefig(path)


def plot_mlp_curves(h, path, title):
    ep = np.arange(1, len(h["train_err"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(ep, h["train_err"], label="train"); ax[0].plot(ep, h["val_err"], label="validation")
    ax[0].set_title(f"{title}: error vs epochs"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("error rate")
    ax[1].plot(ep, h["train_loss"], label="train"); ax[1].plot(ep, h["val_loss"], label="validation")
    ax[1].set_title(f"{title}: loss vs epochs"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("loss")
    for a in ax:
        a.legend()
    savefig(path)


def plot_overfitting(h_reg, h_unreg, path):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for a, h, t in zip(ax, (h_unreg, h_reg), ("No regularisation", "Tuned (dropout / weight decay)")):
        ep = np.arange(1, len(h["train_err"]) + 1)
        a.plot(ep, h["train_err"], label="train error"); a.plot(ep, h["val_err"], label="val error")
        a.set_title(t); a.set_xlabel("epoch"); a.legend()
    ax[0].set_ylabel("error rate")
    savefig(path)


def plot_ab(results, pla_hist, mlp_hist, path_bar, path_curve, path_roc):
    metrics = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "auc_macro"]
    x = np.arange(len(metrics)); w = 0.35
    plt.figure(figsize=(8, 4.5))
    for k, r in enumerate(results):
        plt.bar(x + (k - 0.5) * w, [r[m] for m in metrics], w, label=r["model"])
    plt.xticks(x, [m.replace("_", "\n") for m in metrics]); plt.ylim(0, 1); plt.legend()
    plt.title("A/B comparison on the test set")
    savefig(path_bar)

    plt.figure(figsize=(7, 4.5))
    plt.plot(np.arange(1, len(pla_hist["train_err"]) + 1), pla_hist["train_err"], label="PLA train")
    plt.plot(np.arange(1, len(pla_hist["val_err"]) + 1), pla_hist["val_err"], "--", label="PLA val")
    plt.plot(np.arange(1, len(mlp_hist["train_err"]) + 1), mlp_hist["train_err"], label="MLP train")
    plt.plot(np.arange(1, len(mlp_hist["val_err"]) + 1), mlp_hist["val_err"], "--", label="MLP val")
    plt.xlabel("epoch"); plt.ylabel("error rate"); plt.legend()
    plt.title("Convergence: PLA vs tuned MLP")
    savefig(path_curve)

    plt.figure(figsize=(6.5, 5))
    for r in results:
        plt.plot(r["_micro"][0], r["_micro"][1], label=f"{r['model']} micro (AUC={r['_micro'][2]:.3f})")
        plt.plot(r["_macro"][0], r["_macro"][1], "--", label=f"{r['model']} macro (AUC={r['_macro'][2]:.3f})")
    plt.plot([0, 1], [0, 1], "k:", lw=0.8)
    plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
    plt.title("ROC comparison"); plt.legend(loc="lower right", fontsize=8)
    savefig(path_roc)

In [15]:
def build_observations(res_pla, res_mlp, pla_hist, mlp_hist, unreg_hist, unreg_test_acc,
                       impact, stage_df, stages, cache, base, final_cfg, args):
    L = []
    L.append("OBSERVATIONS (auto-generated from this run - rewrite in your own words in the report)\n")

    L.append("1. Why does PLA underperform compared to MLP?")
    L.append(f"   Test accuracy: PLA = {res_pla['accuracy']:.4f}, MLP = {res_mlp['accuracy']:.4f}; "
             f"macro-F1: PLA = {res_pla['f1_macro']:.4f}, MLP = {res_mlp['f1_macro']:.4f}.")
    L.append(f"   PLA still updated weights on {pla_hist['n_updates'][-1]} training samples in its last epoch "
             f"(final train error {pla_hist['train_err'][-1]:.4f}) -> the 62 classes are not linearly separable "
             f"in pixel space. PLA has one linear boundary per class and a step activation (no gradient, no "
             f"feature learning); the MLP's hidden layers with non-linear activations learn intermediate "
             f"features and non-linear boundaries via back-propagation.\n")

    L.append("2. Which hyper-parameters had the most impact?")
    L.append("   Random-search impact ranking (spread of mean validation accuracy across levels):")
    for _, r in impact.iterrows():
        L.append(f"     {r['hyperparameter']:<11s} spread={r['mean_val_acc_range']:.4f}  "
                 f"best level={r['best_level']}  worst level={r['worst_level']}")
    L.append("   Greedy-stage spread (max - min validation accuracy inside each stage):")
    for st, g in stage_df.groupby("stage"):
        L.append(f"     {st:<22s} {g['val_acc'].max() - g['val_acc'].min():.4f}")
    L.append("")

    L.append("3. Did optimizer choice (SGD vs Adam) affect convergence?")
    grid = stages["optimizer_lr_grid"]
    overall_best = max(grid.values())
    target = 0.9 * overall_best
    for opt in SPACE["optimizer"]:
        lr = max(SPACE["lr"], key=lambda l: grid[(opt, l)])
        _, h = cache[cfg_key({**base, "optimizer": opt, "lr": lr})]
        va = 1 - np.array(h["val_err"])
        hit = np.argmax(va >= target) + 1 if (va >= target).any() else None
        L.append(f"     {opt:<13s} best lr={lr:<7g} best val acc={grid[(opt, lr)]:.4f}  "
                 f"final train loss={h['train_loss'][-1]:.4f}  "
                 f"epochs to reach 90% of overall best val acc: {hit if hit else 'not reached'}")
    L.append("   (full-batch GD makes one update per epoch, so it needs far more epochs than SGD/Adam.)\n")

    L.append("4. Did adding more hidden layers always improve results?")
    for lab, acc in stages["hidden"]:
        L.append(f"     hidden={lab:<16s} val acc={acc:.4f}")
    L.append("   Look for a plateau/drop: deeper nets have more parameters on only ~2.4k training images "
             "(overfit), and harder optimisation (vanishing gradients with sigmoid/tanh) under a fixed "
             "epoch budget.\n")

    L.append("5. Did the MLP overfit? How can it be mitigated?")
    tr_acc = 1 - mlp_hist["train_err"][int(np.argmax(1 - np.array(mlp_hist['val_err'])))]
    va_acc = max(1 - np.array(mlp_hist["val_err"]))
    L.append(f"   Tuned MLP: train acc at best epoch = {tr_acc:.4f}, val acc = {va_acc:.4f}, "
             f"test acc = {res_mlp['accuracy']:.4f} (train-val gap = {tr_acc - va_acc:.4f}).")
    ur_tr = 1 - unreg_hist["train_err"][int(np.argmax(1 - np.array(unreg_hist['val_err'])))]
    ur_va = max(1 - np.array(unreg_hist["val_err"]))
    L.append(f"   Same config without dropout/weight decay: train acc = {ur_tr:.4f}, val acc = {ur_va:.4f}, "
             f"test acc = {unreg_test_acc:.4f} (gap = {ur_tr - ur_va:.4f}).")
    L.append("   Mitigation: dropout, L2 weight decay, early stopping (used: best-val checkpoint), smaller "
             "network, data augmentation (small shifts/rotations), more data.\n")

    L.append("FINAL MLP CONFIGURATION")
    for k, v in final_cfg.items():
        L.append(f"     {k}: {hid_label(v) if k == 'hidden' else v}")
    return "\n".join(L)

In [ ]:
def main():
    ap = argparse.ArgumentParser()

    ap.add_argument(
    "--data_dir",
    default="/root/.cache/kagglehub/datasets/dhruvildave/english-handwritten-characters-dataset/versions/3"
)
    ap.add_argument("--out_dir", default="/content/exp9_outputs")
    ap.add_argument("--img_size", type=int, default=32)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--pla_epochs", type=int, default=100)
    ap.add_argument("--pla_lr", type=float, default=0.1)
    ap.add_argument("--trials", type=int, default=24)
    ap.add_argument("--tune_epochs", type=int, default=30)
    ap.add_argument("--final_epochs", type=int, default=100)
    ap.add_argument("--quick", action="store_true")

    args = ap.parse_args(args=[])

    if args.quick:
        args.pla_epochs = 10
        args.trials = 4
        args.tune_epochs = 5
        args.final_epochs = 15

    os.makedirs(args.out_dir, exist_ok=True)
    out = lambda f: os.path.join(args.out_dir, f)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(args.seed)
    print(f"Device: {device}")

    # ---------------- Preprocessing ----------------
    print("\n[1] Loading & preprocessing")
    X, y, classes = load_dataset(args.data_dir, args.img_size)
    C = len(classes)
    print(f"  {X.shape[0]} images, {X.shape[1]} features, {C} classes")
    save_sample_grid(X, y, classes, args.img_size, out("samples.png"))
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=args.seed)
    X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp,
                                              random_state=args.seed)
    mu, sd = X_tr.mean(0), X_tr.std(0) + 1e-6          # statistics from the TRAIN split only
    X_tr, X_va, X_te = [((a - mu) / sd).astype(np.float32) for a in (X_tr, X_va, X_te)]
    print(f"  split: train={len(X_tr)}  val={len(X_va)}  test={len(X_te)}  (stratified 70/15/15)")

    # ---------------- Model A: PLA ----------------
    print("\n[2] Model A: Perceptron Learning Algorithm")
    pla = PerceptronOvR(C, lr=args.pla_lr, epochs=args.pla_epochs, seed=args.seed).fit(X_tr, y_tr, X_va, y_va)
    plot_pla_convergence(pla.history, out("pla_convergence.png"))
    pla_scores = pla.decision_function(X_te)
    res_pla = evaluate_model("PLA", y_te, pla_scores.argmax(1), pla_scores, classes, args.out_dir)
    print(f"  PLA test accuracy = {res_pla['accuracy']:.4f}")

    # ---------------- Model B: MLP tuning ----------------
    print("\n[3] Model B: MLP hyper-parameter tuning")
    T = lambda a, dt: torch.tensor(a, dtype=dt, device=device)
    D = {"Xtr": T(X_tr, torch.float32), "ytr": T(y_tr, torch.long),
         "Xva": T(X_va, torch.float32), "yva": T(y_va, torch.long),
         "Xte": T(X_te, torch.float32), "yte": T(y_te, torch.long), "n_classes": C}
    cache = {}

    print("\n  (a) Random search over the full space")
    rs = random_search(D, args.trials, args.tune_epochs, device, args.seed, cache)
    rs_df = pd.DataFrame([r for _, r in rs]).sort_values("val_acc", ascending=False)
    rs_df.to_csv(out("random_search_results.csv"), index=False)
    impact = hyperparameter_impact(rs_df)
    impact.to_csv(out("hyperparameter_impact.csv"), index=False)
    plot_impact(impact, out("hyperparameter_impact.png"))
    best_rs_cfg = rs[int(np.argmax([r["val_acc"] for _, r in rs]))][0]
    print(f"\n  Best random-search config: {best_rs_cfg}")

    print("\n  (b) Greedy stage-wise tuning starting from the best random-search config")
    base, stages, stage_df = greedy_stage_tuning(best_rs_cfg, D, args.tune_epochs, device, args.seed, cache)
    stage_df.to_csv(out("tuning_stages.csv"), index=False)
    plot_optimizer_lr_heatmap(stages["optimizer_lr_grid"], out("tuning_optimizer_lr.png"))
    plot_stage_bars(stages, out("tuning_stages.png"))
    plot_optimizer_convergence(cache, base, stages["optimizer_lr_grid"], out("optimizer_convergence.png"))

    final_cfg = base
    print(f"\n  FINAL CONFIG: {final_cfg}")
    with open(out("best_hyperparameters.json"), "w") as fh:
        json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in final_cfg.items()}, fh, indent=2)

    # ---------------- Final training + test evaluation ----------------
    print("\n[4] Training final MLP (best-validation checkpoint) and evaluating on the test set")
    model, hist, best_acc, best_ep = train_mlp(final_cfg, D, args.final_epochs, device, args.seed)
    print(f"  best val acc = {best_acc:.4f} at epoch {best_ep}")
    plot_mlp_curves(hist, out("mlp_convergence.png"), "Tuned MLP")
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(D["Xte"]), dim=1).cpu().numpy()
    res_mlp = evaluate_model("MLP", y_te, probs.argmax(1), probs, classes, args.out_dir)
    print(f"  MLP test accuracy = {res_mlp['accuracy']:.4f}")

    # Overfitting study: same config with / without regularisation
    unreg_cfg = {**final_cfg, "dropout": 0.0, "weight_decay": 0.0}
    m2, hist_un, _, _ = train_mlp(unreg_cfg, D, args.final_epochs, device, args.seed)
    m2.eval()
    with torch.no_grad():
        unreg_test_acc = (m2(D["Xte"]).argmax(1) == D["yte"]).float().mean().item()
    plot_overfitting(hist, hist_un, out("overfitting_comparison.png"))

    # ---------------- A/B comparison ----------------
    print("\n[5] A/B comparison")
    results = [res_pla, res_mlp]
    table = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in results])
    table.to_csv(out("ab_comparison.csv"), index=False)
    print(table.round(4).to_string(index=False))
    plot_ab(results, pla.history, hist, out("ab_metrics.png"), out("ab_convergence.png"),
            out("ab_roc.png"))

    obs = build_observations(res_pla, res_mlp, pla.history, hist, hist_un, unreg_test_acc, impact,
                             stage_df, stages, cache, base, final_cfg, args)
    with open(out("observations.txt"), "w") as fh:
        fh.write(obs)
    print("\n" + obs)
    print(f"\nDone. Everything is in: {os.path.abspath(args.out_dir)}")


if __name__ == "__main__":
    main()

Device: cpu

[1] Loading & preprocessing
  3410 images, 1024 features, 62 classes
  saved /content/exp9_outputs/samples.png
  split: train=2387  val=511  test=512  (stratified 70/15/15)

[2] Model A: Perceptron Learning Algorithm
    PLA epoch   1/100  train_err=0.8869  val_err=0.9315  samples_updated=2374
    PLA epoch  10/100  train_err=0.4801  val_err=0.8571  samples_updated=1895
    PLA epoch  20/100  train_err=0.3712  val_err=0.8219  samples_updated=1601
    PLA epoch  30/100  train_err=0.3159  val_err=0.8395  samples_updated=1432
    PLA epoch  40/100  train_err=0.2329  val_err=0.8258  samples_updated=1277
    PLA epoch  50/100  train_err=0.2061  val_err=0.8180  samples_updated=1188
    PLA epoch  60/100  train_err=0.1940  val_err=0.8376  samples_updated=1042
    PLA epoch  70/100  train_err=0.1487  val_err=0.8297  samples_updated=949
    PLA epoch  80/100  train_err=0.1382  val_err=0.8337  samples_updated=840
    PLA epoch  90/100  train_err=0.1093  val_err=0.8258  samples_updat